In [20]:
import sys
from typing import Literal

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import display
from scipy.stats import genpareto, lognorm, truncpareto

from pandemic_model.stats.mevd import mevd_gen
from pandemic_model.stats.pareto import fit_trunc_pareto

sys.path.append("../scripts")
from fit_distributions import fit_truncated_pareto

In [21]:
all_epidemics_ds = pd.read_excel("../../data/raw/epidemics_marani_240816.xlsx")
all_epidemics_ds = all_epidemics_ds.rename(columns={'severity_smu': 'severity'}) # Name change simplifies code later

In [22]:
with open("../../data/clean/inverted_covid_severity.yaml", 'rb') as f:
  inverted_covid_severity_dict = yaml.safe_load(f)
  inverted_covid_severity = inverted_covid_severity_dict['ex_ante_severity']

In [23]:
all_epidemics_ds['intensity'] = all_epidemics_ds['severity'] / all_epidemics_ds['duration']

#### Construct datasets

In [24]:
bernstein_additions = ['hiv/aids', 'covid-19'] # Anything else we added?
marani_raw_ds = all_epidemics_ds[~all_epidemics_ds['disease'].isin(bernstein_additions)]
marani_fit_ds = marani_raw_ds[marani_raw_ds['year_end'].between(1600, 1945)]
all_modern_ds = all_epidemics_ds[all_epidemics_ds['year_start'] >= 1900]
modern_viral_ds = all_modern_ds[all_modern_ds['type'].str.contains('viral', case=False)]

In [25]:
modern_viral_ds['disease'].value_counts().sort_values(ascending=False)

disease
polio                         19
influenza                     18
smallpox                      13
meningitis                    11
measles                        9
dengue                         9
encephalitis                   8
yellow fever                   6
pneumonia                      5
murray valley encephalitis     4
ebola                          3
west nile                      2
rubella                        2
mumps                          1
kyasanur forest disease        1
hemorrhagic fever              1
rift valley fever              1
hiv/aids                       1
sars                           1
mers                           1
covid-19                       1
Name: count, dtype: int64

In [26]:
# Not sure about these
recurring_viruses = ['polio', 'smallpox', 'meningitis', 'measles', 'dengue',
                     'encephalitis', 'yellow fever', 'pneumonia', 'murray valley encephalitis',
                     'rubella', 'mumps']
non_resp_novel = ['ebola', 'west nile', 'kyasanur forest disease', 'hemorrhagic fever', 'rift valley fever', 'hiv/aids']

In [27]:
modern_viral_no_reccuring = modern_viral_ds[~modern_viral_ds['disease'].isin(recurring_viruses)]
modern_resp_novel = modern_viral_no_reccuring[~modern_viral_no_reccuring['disease'].isin(non_resp_novel)]

In [28]:
bernstein_intersect_ds = pd.read_excel("../../data/raw/novel_resp_241228.xlsx")
bernstein_intersect_ds = bernstein_intersect_ds.rename(columns={'severity_smu': 'severity'})

In [29]:
fit_dfs = {
		'All 1600-1945 (Marani)': marani_fit_ds,
		'All since 1900': all_modern_ds, 
		'Viral since 1900': modern_viral_ds,
		'Viral since 1900, no recurring': modern_viral_no_reccuring,
		'Respiratory since 1900': modern_resp_novel,
		'Novel since 1900 (Bernstein)': bernstein_intersect_ds
}

### Fit distributions

In [30]:
def get_pareto_dist(df: pd.DataFrame,
                    col: Literal['severity', 'intensity'],
                    disttype: Literal['gpd', 'trunc'],
                    lower_bound: float = 1e-2,
                    upper_bound: float = 1e4):
		"""Get exceedance distribution."""
		fit_df = df.copy()
		
		# Drop observations below lower bound
		fit_df = fit_df[fit_df[col] >= lower_bound]
				
		if disttype == 'gpd':
				c, loc, scale = genpareto.fit(fit_df[col], floc=lower_bound)
				dist = genpareto(c=c, loc=loc, scale=scale)                         
		else:
				(b, loc, scale), _ = fit_truncated_pareto(fit_df[col], lower_bound, upper_bound)
				c = (upper_bound - loc) / scale
				dist = truncpareto(b=b, c=c, loc=loc, scale=scale)

		return dist

In [31]:
def get_datasets_and_distributions(lower_bound=1e-2,
                                   drop_hiv=False,
                                   use_inverted_covid=False):
    """
    Creates plots comparing exceedance functions between generalized and truncated Pareto fits
    for both severity and intensity data.
    
    Args:
        lower_bound (float): Lower bound threshold for fitting distributions
        drop_hiv (bool): Whether to exclude HIV/AIDS observation
        use_inverted_covid (bool): Whether to use inverted COVID-19 severity
    
    Returns:
        dict: Dictionary containing the processed dataframes and fitted distributions
    """
    # Should probably define elsewhere
    upper_bounds = {
      'severity': {
        'gpd': None,
        'trunc': 1e4
			},
      'intensity': {
        'gpd': None,
        'trunc': 1e4
			}
		}
    
    def preprocess_df(df, drop_hiv=drop_hiv, use_inverted_covid=use_inverted_covid):
        new_df = df.copy()
        
        if drop_hiv:
            new_df = new_df[new_df['disease'] != 'hiv/aids']
        
        if use_inverted_covid:
            covid_mask = new_df['disease'] == 'covid-19'
            new_df.loc[covid_mask, 'severity'] = inverted_covid_severity
        
        new_df['intensity'] = new_df['severity'] / new_df['duration']
        return new_df
  
    # Process all dataframes
    processed_dfs = {name: preprocess_df(df) for name, df in fit_dfs.items()}
    
    # Fit distributions
    distributions = {}
    for name, df in processed_dfs.items():
        distributions[name] = {
            metric: {
                dist_type: get_pareto_dist(
                    df,
                    metric,
                    dist_type,
                    lower_bound,
                    upper_bounds[metric][dist_type]
                )
                for dist_type in ['gpd', 'trunc']
            }
            for metric in ['severity', 'intensity']
        }
    
    return {'dfs': processed_dfs, 'distributions': distributions}

In [32]:
# Get arrival counts for MEVD
def get_annual_arrival_counts(df: pd.DataFrame, period_start, period_end):
		years = np.arange(period_start, period_end + 1)
		arrival_counts = pd.Series(0, index=years)

		eval_df = df[df['year_start'].between(period_start, period_end)]
		df_counts = eval_df.groupby('year_start').size()
		arrival_counts.loc[df_counts.index] = df_counts

		return arrival_counts


arrival_count_dfs = {
		'All 1600-1945 (Marani)': marani_raw_ds,
		'All since 1900': all_modern_ds, 
		'Viral since 1900': modern_viral_ds,
		'Viral since 1900, no recurring': modern_viral_no_reccuring,
		'Respiratory since 1900': modern_resp_novel,
		'Novel since 1900 (Bernstein)': bernstein_intersect_ds
}
		
long_arrival_windows = {
		'All 1600-1945 (Marani)': [1600, 1945],
		'All since 1900': [1900, 2019], 
		'Viral since 1900': [1900, 2019],
		'Viral since 1900, no recurring': [1900, 2019],
		'Respiratory since 1900': [1900, 2019],
		'Novel since 1900 (Bernstein)': [1900, 2019]
}

In [33]:
def get_share_above_threshold(df, col, thresh):
    return (
        len(df[df[col] >= thresh]) / len(df)
        if len(df) > 0 
        else 0
    )

In [34]:
# Interactive plotting code
def create_interactive_plot():
    """Creates and displays the interactive plot with all controls."""
    
    def update_plot(
      	lower_bound,
        show_gpd,
        show_trunc,
        use_recent_arrival_counts,
        use_above_threshold_rate,
        selected_dfs,
        use_extinction_upper,
        drop_hiv,
        use_inverted_covid
    ):
        """Updates and returns the plot based on the selected parameters.
        
        Args:
            lower_bound (float): Lower bound for fitting distributions
            show_gpd (bool): Whether to show generalized Pareto distribution curves
            show_trunc (bool): Whether to show truncated distribution curves 
            use_recent_arrival_counts (bool): Whether to use short or long run arrival counts.
            use_above_threshold_rate (bool): Whether to adjust rates for above threshold events
            selected_dfs (list): List of selected datasets to plot
            use_extinction_upper (bool): Whether to use extinction upper bounds
            drop_hiv (bool): Whether to exclude HIV data
            use_inverted_covid (bool): Whether to use inverted COVID data
            
        Returns:
            matplotlib.figure.Figure: The updated figure
        """
        # Get initial datasets and distributions
        out = get_datasets_and_distributions(
						lower_bound=lower_bound,
						drop_hiv=drop_hiv,
						use_inverted_covid=use_inverted_covid
				)
        processed_dfs = out['dfs']
        distributions = out['distributions']

        fig, (ax_sev, ax_int) = plt.subplots(1, 2, figsize=(16, 8))
        
        # Generate points for theoretical curves
        x = np.logspace(np.log10(lower_bound), np.log10(1e4), 1000)
        
        for name in selected_dfs:
            df = processed_dfs[name]
            
						# Bad style due to silly tab errors
            arrival_window = [2000, 2019] if use_recent_arrival_counts else long_arrival_windows[name]
            arrival_count_df = arrival_count_dfs[name]
            arrival_counts = get_annual_arrival_counts(arrival_count_df, *arrival_window)

            # Plot for both severity and intensity
            for ax, metric in [(ax_sev, 'severity'), (ax_int, 'intensity')]:
                
                share_above_threshold = (
                    get_share_above_threshold(df, metric, lower_bound)
                    if use_above_threshold_rate
                    else 1.0
								)
                            
                if show_gpd:
                    base_dist = distributions[name][metric]['gpd']
                    mevd_gpd = mevd_gen(arrival_counts, base_dist)
                    y_gpd = mevd_gpd._sf(x) * share_above_threshold # Check if this is correct.
                    # Set to zero above upper bound
                    max_level = 171 if metric == 'severity' else 57
                    max_level = 1e4 if use_extinction_upper else max_level
                    y_gpd[x > max_level] = 0
                    ax.semilogx(x, y_gpd, label=f'{name} (GPD)', linestyle='-')
                
                if show_trunc:
                    base_dist = distributions[name][metric]['trunc']
                    mevd_trunc = mevd_gen(arrival_counts, base_dist)
                    y_trunc = mevd_trunc._sf(x) * share_above_threshold
                    ax.semilogx(x, y_trunc, label=f'{name} (Truncated)', linestyle='--')
        
        # Configure axes
        ax_int.legend(loc='upper right')

        for ax, title in [(ax_sev, 'Severity'), (ax_int, 'Intensity')]:
            ax.set_ylim(0, 0.4)
            ax.set_xlim(1e-3, 1e4)
            ax.set_xlabel(f'{title} (deaths per 10k)')
            ax.set_ylabel('Exceedance probability (per year)')
            ax.grid(True, which='both', ls='-', alpha=0.2)
        
        plt.tight_layout()
        return fig
    
    # Create widgets
    lower_bound_slider = widgets.FloatLogSlider(
        value=1e-2, min=-3, max=1, description='Lower bound:'
    )
    
    show_gpd = widgets.Checkbox(
        value=True, description='Show GPD'
    )
    
    show_trunc = widgets.Checkbox(
        value=True, description='Show Truncated'
    )
    
    use_recent_counts = widgets.Checkbox(
        value=True, description='Use recent arrival counts'
    )
    
    use_above_threshold = widgets.Checkbox(
        value=False, description='Adjust for above threshold reate'
    )
    
    use_extinction_upper = widgets.Checkbox(
        value=False, description='Use extinction upper bounds'
    )
    
    drop_hiv = widgets.Checkbox(
        value=False, description='Drop HIV'
    )
    
    use_inverted_covid = widgets.Checkbox(
        value=False, description='Use inverted COVID'
    )
    
    df_toggles = widgets.SelectMultiple(
        options=[
						'All 1600-1945 (Marani)',
						'All since 1900',
						'Viral since 1900',
						'Viral since 1900, no recurring',
						'Respiratory since 1900',
						'Novel since 1900 (Bernstein)',
        ],
        value=['All 1600-1945 (Marani)'],
        description='Datasets:'
    )
    
    # Create interactive plot
    interactive_plot = widgets.interactive(
        update_plot,
        lower_bound=lower_bound_slider,
        show_gpd=show_gpd,
        show_trunc=show_trunc,
        use_recent_arrival_counts=use_recent_counts,
        use_above_threshold_rate=use_above_threshold,
        selected_dfs=df_toggles,
        use_extinction_upper=use_extinction_upper,
        drop_hiv=drop_hiv,
        use_inverted_covid=use_inverted_covid
    )
    
    # Layout widgets and plot
    controls = widgets.VBox([
        df_toggles,
        widgets.HBox([lower_bound_slider, show_gpd, show_trunc, use_recent_counts]),
        widgets.HBox([use_above_threshold, use_extinction_upper, drop_hiv, use_inverted_covid])
    ])
    
    return widgets.VBox([controls, interactive_plot.children[-1]])


In [37]:
bernstein_intersect_ds

,location,year_start,year_end,duration,death_thousand,pop_thousand,severity_perthousand,severity,disease,type,transmission,is_vira_only,is_vira_mixed,contains_vira,is_pandemic
0,pandemic spanish flu,1918,1920,3,32000.000,1873300.000,17.082154,170.821545,influenza,viral,droplet,1,0,1,1
1,pandemic of asian flu,1957,1958,2,2000.000,2873306.000,0.696062,6.960623,influenza,viral,droplet,1,0,1,1
2,hong kong flu,1968,1969,2,1000.000,3551599.000,0.281563,2.815633,influenza,viral,droplet,1,0,1,1
3,hiv/aids pandemic,1981,2024,44,42300.000,4536996.619,9.323348,93.233484,hiv/aids,viral,fluid,1,0,1,1
4,global sars,2003,2003,1,0.744,6381185.141,0.000117,0.001166,sars,viral,droplet,1,0,1,1
5,swine flu,2009,2009,1,284.500,6872767.000,0.041395,0.413953,influenza,viral,droplet,1,0,1,1
6,global mers,2012,2017,6,0.659,7125827.957,0.000092,0.000925,mers,viral,droplet,1,0,1,1
7,western africa,2013,2016,4,11.325,7210582.000,0.001571,0.015706,ebola,viral,blood/fluid,1,0,1,1
8,global covid-19,2019,2024,6,7100.000,7740000.000,0.917313,9.173127,covid-19,viral,airborne/droplet,1,0,1,1


In [35]:
create_interactive_plot()